In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
np.seterr(divide='ignore', invalid='ignore')

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *
from domain_funcs import *

dpath0='/discover/nobackup/projects/giss/baldwin_nip/dmkumar' # top level data directory

In [2]:
### +++ DATA PATHS +++ ###

obs_prods=['imerg']
cases=['pi','2xco2']

files={ 'obs':    {}, 
        'ctrl':   { 'pi':{}, '2xco2':{} },
        'hitopo': { 'pi':{}, '2xco2':{} } }

for key in ['obs']:
    for i,run in enumerate(obs_prods):
        files[key][run] = f'{dpath0}/obs_data/prec/imerg.gn.timeseries.2001-2018.nc' 

for key in ['ctrl','hitopo']:
    varn='precip'
    for i,case in enumerate(cases):
        if case=='pi':
            files[key][case] = f'{dpath0}/FLOR/{key}/{case}/flor.{key}.{varn}.monthly.nc'
        elif case=='2xco2':
            files[key][case] = f'{dpath0}/FLOR/{key}/{case}/flor.{key}.{case}.{varn}.monthly.nc'

for key in ['topo']:
    files[key] = {}
    files[key]['etopo'] = f'{dpath0}/topo_files/obs.etopo5.zsurf.nc'
    files[key]['ctrl'] = f'{dpath0}/topo_files/flor.ctrl.zsurf.nc'
    files[key]['hitopo'] = f'{dpath0}/topo_files/flor.hitopo.zsurf.nc'

In [3]:
## +++ ORGANIZE DATA +++ ##

# --- sub-setting bounds --- #
latmin, latmax = 10, 70
lonmin, lonmax = 200, 300
n_years = -100
n_keep  = n_years * 12 # number of time steps to keep in months

# initialize dictionaries
dat={ 'obs':    {}, 
      'ctrl':   { 'pi':{}, '2xco2':{} },
      'hitopo': { 'pi':{}, '2xco2':{} } }

print('Working on...')
    
for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        ds = xr.open_dataset(files[key][run], chunks={}).precipitation * 24 # convert from mm/hr to mm/day
        ds = ds.transpose('time','lat','lon')
        ds.attrs['units'] = 'mm/day'
        ds.attrs['Units'] = 'mm/day'
        ds = lonFlip(ds) # switch lons from -180:180 to 0:360
        dat[key][run] = ds.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
        del ds
        
for key in ['ctrl','hitopo']:
    print(f'{key}')
    for case in cases:
        ds = xr.open_dataset(files[key][case]).precip[n_keep:,:,:] * 86400 
        ds = ds.rename({'grid_xt':'lon','grid_yt':'lat'}) # update coordinate names to match imerg
        ds.attrs['units'] = 'mm/day' # update units
        dat[key][case] = ds.sel(lon=slice(lonmin,lonmax), lat=slice(latmin,latmax))
        del ds

topo = {}
for key in ['topo']:
    for case in ['ctrl', 'hitopo']:
        ds = xr.open_dataset(files[key][case]).ZSURF.rename({'GRID_XT':'lon', 'GRID_YT':'lat'})
        ds_flip = lonFlip(ds)
        topo[case] = ds_flip.where(ds_flip>0, np.nan).sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
        del ds
        del ds_flip

print('Done.')

Working on...
obs
ctrl
hitopo
Done.


In [8]:
### +++ CALCULATE SEASONAL TIME-MEANS +++ ###

jas_mean={ 'obs':    {}, 
           'ctrl':   { 'pi':{}, '2xco2':{} },
           'hitopo': { 'pi':{}, '2xco2':{} } }
ann_jas_mean={ 'obs':    {}, 
               'ctrl':   { 'pi':{}, '2xco2':{} },
               'hitopo': { 'pi':{}, '2xco2':{} } }

print('Calculating seasonal means for...')
for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        jas_mean[key][run]     = jas_seasonal_mean(dat[key][run])
        ann_jas_mean[key][run] = jas_yearly_mean(dat[key][run])
        
for key in ['ctrl','hitopo']:
    print(f'{key}')
    for case in cases:
        jas_mean[key][case]     = jas_seasonal_mean(dat[key][case])
        ann_jas_mean[key][case] = jas_yearly_mean(dat[key][case])

print('Done.')

Calculating seasonal means for...
obs
ctrl


AttributeError: 'dict' object has no attribute 'sel'

In [13]:
## +++ COMPARING 2xCO2 MODEL RUNS TO MODEL PI +++ ##

future_diff      = { 'ctrl':{}, 'hitopo':{} }
future_diff_mask = { 'ctrl':{}, 'hitopo':{} }
future_ptvals    = { 'ctrl':{}, 'hitopo':{} }

print('Significance testing for:')
for key in ['ctrl','hitopo']:
    print(f'{key}')
    diff_, diff_mask_, ptvals_ = sigtest(ann_jas_mean[key]['2xco2'], ann_jas_mean[key]['pi'],
                                         jas_mean[key]['2xco2'],     jas_mean[key]['pi'])
    future_diff[key] = diff_
    future_diff_mask[key] = diff_mask_
    future_ptvals[key] = ptvals_
print('Done.')


Significance testing for:
ctrl
hitopo
Done.


In [14]:
coords_map, regions = nam_regions()
domains = list(coords_map.keys())

## FIGS

### 2xCO2$-$PI absolute change in JAS precipitation

In [ ]:
# --- Settings --- #
text_kw={'color':'k', 'weight':'bold', 'size':24, 'horizontalalignment':'center', 'verticalalignment':'bottom'}
text_kw2={'color':'k', 'weight':'normal', 'size':14, 'horizontalalignment':'left', 'verticalalignment':'bottom'}
text_kw3={'color':'k', 'weight':'bold', 'size':30, 'horizontalalignment':'center', 'verticalalignment':'bottom'}
titles=([r'CTRL', r'$\mathbf{HI_{gbl}}$', r'B$-$A'])
letters=['A','B','C']
tx=-109
ty=38.1
# bias colormap
dcmap,_,_,_=get_settings(field='precip', diff=True)
dvmin=-2
dvmax=2
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(800,3800,7)
dzlevels=np.linspace(400,1600,5)
# stippling
step=1
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[243,259,18,38]

# --- Define vars for plotting --- #
future_diff_diff = future_diff['hitopo'] - future_diff['ctrl']

lon=jas_mean['ctrl']['pi'].lon
lat=jas_mean['ctrl']['pi'].lat
lon2d, lat2d = np.meshgrid(lon, lat)

# --- plot data --- #    
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20,10), layout='constrained', subplot_kw={'projection':proj})

# prec change
for i,run in enumerate(future_diff.keys()):
    sig_mask = future_diff_mask[run].isnull().values
    ax[i].pcolormesh(lon, lat, future_diff[run], cmap=dcmap, norm=dnorm, transform=trans)
    ax[i].scatter(lon2d[sig_mask][::step], lat2d[sig_mask][::step],
                  s=30, c='k', marker='.', linewidths=0, transform=trans, zorder=100)
    # topo contours
    ax[i].contour(lon, lat, topo[run], levels=zlevels, linewidths=1.5, colors='black', transform=trans)


# prec change difference HIgbl - CTRL
cf2=ax[2].pcolormesh(lon, lat, future_diff_diff, cmap=dcmap, norm=dnorm, transform=trans) 
ax[2].contour(lon, lat, topo['hitopo']-topo['ctrl'], levels=dzlevels, linewidths=1.5, colors='black', transform=trans)

# --- formatting --- #
for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-117, 38.25, letters[i], **text_kw3)
    # boxes around NAM subdomains
    poly1=patches.Polygon(coords_map['south'], closed=True, ec='red', fc='none', lw=3, ls='--', transform=ccrs.PlateCarree(), zorder=100); ax.add_patch(poly1)
    poly2=patches.Polygon(coords_map['north'], closed=True, ec='firebrick', fc='none', lw=3, ls='--', transform=ccrs.PlateCarree(), zorder=100); ax.add_patch(poly2)
    # map properties
    ax.coastlines(color='k', lw=1)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=0.5, linestyle='--', zorder=10, draw_labels=True)
    gl.xlocator = mticker.MultipleLocator(4)
    gl.ylocator = mticker.MultipleLocator(3)
    gl.bottom_labels=True; gl.left_labels=(i==0); gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':16}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':16}
    
# --- colorbar --- #
cax=fig.add_axes([1.01, .1, 0.02, 0.8])
cbar=fig.colorbar(cf2, ticks=np.linspace(dvmin,dvmax,11), orientation='vertical', extend='both', cax=cax)
cbar.set_label('2xCO$_2$$-$PI $\Delta$ Precipitation Rate [mm day$^{-1}$]', labelpad=40, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig('../figs/prec_flor_2xco2_abs_change.png', transparent=False, bbox_inches='tight')
#plt.savefig('../figs/prec_flor_2xco2_abs_change.pdf', transparent=False, bbox_inches='tight')

### 2xCO2-PI change in JAS precipitation as percent of seasonal rainfall rate

In [ ]:
# --- Settings --- #
text_kw={'color':'k', 'weight':'bold', 'size':24, 'horizontalalignment':'center', 'verticalalignment':'bottom'}
text_kw2={'color':'k', 'weight':'normal', 'size':14, 'horizontalalignment':'left', 'verticalalignment':'bottom'}
text_kw3={'color':'k', 'weight':'bold', 'size':30, 'horizontalalignment':'center', 'verticalalignment':'bottom'}
titles=([r'CTRL', r'$\mathbf{HI_{gbl}}$', r'B$-$A'])
letters=['A','B','C']
tx=-109
ty=38.1
# bias colormap
dcmap,_,_,_=get_settings(field='precip', diff=True)
dvmin=-60
dvmax=60
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(800,3800,7)
dzlevels=np.linspace(400,1600,5)
# stippling
step=1
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[243,259,18,38]

# --- Define vars for plotting --- #
ds1=(future_diff['ctrl']/jas_mean['ctrl']['pi']) * 100
ds2=(future_diff['hitopo']/jas_mean['hitopo']['pi']) * 100
future_diff_diff = ds2-ds1

lon=jas_mean['ctrl']['pi'].lon
lat=jas_mean['ctrl']['pi'].lat
lon2d, lat2d = np.meshgrid(lon, lat)

# --- plot data --- #    
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20,10), layout='constrained', subplot_kw={'projection':proj})

# prec change
for i,run in enumerate(future_diff.keys()):
    pct_change = (future_diff[run]/jas_mean[run]['pi']) * 100
    pct_change_mask = future_diff_mask[run].isnull().values
    # prec change
    ax[i].pcolormesh(lon, lat, pct_change, cmap=dcmap, norm=dnorm, transform=trans)
    ax[i].scatter(lon2d[pct_change_mask][::step], lat2d[pct_change_mask][::step],
                  s=30, c='k', marker='.', linewidths=0, transform=trans, zorder=100 )
    # topo contours
    ax[i].contour(lon, lat, topo[run], levels=zlevels, linewidths=1.5, colors='black', transform=trans)


# prec change difference HIgbl - CTRL
cf2=ax[2].pcolormesh(lon, lat, future_diff_diff, cmap=dcmap, norm=dnorm, transform=trans) 
# !!!!!!!!!!!!!!!!!!!!!!! CHECK HOW THE DASHED LINES LOOK !!!!!!!!!!!!!!!!!!!!!!!
ax[2].contour(lon, lat, topo['hitopo']-topo['ctrl'], levels=dzlevels, linewidths=1.5, ls='--', colors='black', transform=trans)

# --- formatting --- #
for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-117, 38.25, letters[i], **text_kw3)
    # boxes around NAM subdomains
    poly1=patches.Polygon(coords_map['south'], closed=True, ec='red', fc='none', lw=3, ls='--', transform=ccrs.PlateCarree(), zorder=100); ax.add_patch(poly1)
    poly2=patches.Polygon(coords_map['north'], closed=True, ec='firebrick', fc='none', lw=3, ls='--', transform=ccrs.PlateCarree(), zorder=100); ax.add_patch(poly2)
    # map properties
    ax.coastlines(color='k', lw=1)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=0.5, linestyle='--', zorder=10, draw_labels=True)
    gl.xlocator = mticker.MultipleLocator(4)
    gl.ylocator = mticker.MultipleLocator(3)
    gl.bottom_labels=True; gl.left_labels=(i==0); gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':16}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':16}
    
# --- colorbar --- #
cax=fig.add_axes([1.01, .1, 0.02, 0.8])
cbar=fig.colorbar(cf2, ticks=np.linspace(-60,60,9), orientation='vertical', extend='both', cax=cax)
cbar.set_label('(2xCO$_2$ $-$ PI) / PI\n$\Delta$ Relative Precipitation Rate [%]', labelpad=40, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

plt.savefig('../figs/prec_flor_2xco2_pct_change.png', transparent=False, bbox_inches='tight')
plt.savefig('../figs/prec_flor_2xco2_pct_change.pdf', transparent=False, bbox_inches='tight')